# Compilation to the Instruction Set

In this chapter, we will review the steps to convert a circuit to a target ISA defined by `SC_LS_FIXED_V0` (Surface Code - Latice Surgery - Fixed layout version 0) using `qret compile`.
The generated pipeline state JSON will be used to `profile` and visualise the execution on the target machine in the next chapter.

There are four things we discuss in this chapter:

1. Describe the basic execution steps to `compile` the circuit.
2. Show the difference between the `Dim2`, `Dim3`, `DistributedDim2` ISA.
3. Explain how we enable `PBC` (Pauli-based computiation)
4. Provide details about input conditions required for PBC (all qubit measurements at the end)


In [ ]:
import pathlib
import os
import platform

from IPython.display import Code

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
if platform.system() == "Darwin": 
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
else:
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"

os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

## 0. Prior Check
First we make sure we can run `qret` in the CLI.

In [ ]:
!qret --version

## Input Circuit (Normal Mode)
In this chapter the `data/tutorial_5.qasm` circuit will be compiled for all `Dim2`, `Dim3`, and `DistributedDim2` ISA targets.

In [ ]:
tutorial_5_qasm_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5.qasm"
Code(filename=tutorial_5_qasm_path, language="ASM")

## Qret Compile
The `compile` subcommand reads the source code (`IR`, `OpenQASM2`, or `SC_LS_FIXED_V0`), and converts it into an instruction sequence for the target ISA (`SC_LS_FIXED_V0`).

In [ ]:
!qret compile --help

## Reading the Pipeline File
Each mode uses a distinct YAML pipeline file to define specific execution conditions. These include:
- `source`: The language of the input code (IR, OpenQASM2, or the target ISA).
- `input`: The file path to the source code or circuit.
- `output`: The file path for the compiled output.
- `sc_ls_fixed_v0_topology`: The physical topology configuration of the target machine.
- `sc_ls_fixed_v0_machine_type`: The structural architecture variant (Dim2, Dim3, or DistributedDim2).
- `pass`: The list of compiler optimization passes to apply.

Load the file designated for each mode to verify the differences in their configuration settings.

### Machine Type and PBC Mode
Quration supports the following architecture topologies:
- `Dim2`: Single plane
- `Dim3`: 3D placement
- `DistributedDim2`: Distributed quantum computation

Enabling `sc_ls_fixed_v0_enable_pbc_mode` converts the input program into a Pauli-Based Computation (PBC) representation. Note the following constraints:
- Current implementation only supports `Dim2`
- Input circuit requires full qubit measurement at the end

## Compiling to Dim2

In [ ]:
dim2_topology_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim2_topology.yaml"
dim2_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim2_pipeline.yaml"

### Topology Information

In [ ]:
Code(filename=dim2_topology_path, language="YAML")

### Pipeline

In [ ]:
Code(filename=dim2_pipeline_path, language="YAML")

### Execution

In [ ]:
!qret compile --verbose --pipeline {dim2_pipeline_path}
!qret asm -i {output_dir / "tutorial_5_dim2.json"} -o {output_dir / "tutorial_5_dim2.asm"} --print-metadata 1

## Compiling to Dim3

In [ ]:
dim3_topology_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim3_topology.yaml"
dim3_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim3_pipeline.yaml"

### Topology Information

In [ ]:
Code(filename=dim3_topology_path, language="YAML")

### Pipeline

In [ ]:
Code(filename=dim3_pipeline_path, language="YAML")

### Execution

In [ ]:
!qret compile --verbose --pipeline {dim3_pipeline_path}
!qret asm -i {output_dir / "tutorial_5_dim3.json"} -o {output_dir / "tutorial_5_dim3.asm"} --print-metadata 1

## Compilation to Distributed Quantum Computation (DistributedDim2)

In [ ]:
dist_topology_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dist_topology.yaml"
dist_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dist_pipeline.yaml"

### Topology Information

In [ ]:
Code(filename=dist_topology_path, language="YAML")

### Pipeline

In [ ]:
Code(filename=dist_pipeline_path, language="YAML")

### Execution

In [ ]:
!qret compile --verbose --pipeline {dist_pipeline_path}
!qret asm -i {output_dir / "tutorial_5_dist.json"} -o {output_dir / "tutorial_5_dist.asm"} --print-metadata 1

## Compilation to PBC Mode
We enable PBC with `sc_ls_fixed_v0_enable_pbc_mode: true`.
To satisfy the constraints of PBC, the input circuit used will measure all qubits at the end of execution.


In [ ]:
pbc_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_pbc_pipeline.yaml"
pbc_qasm_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_pbc.qasm"

### Input Circuit (for PBC)

In [ ]:
Code(filename=pbc_qasm_path, language="ASM")

### Pipeline

In [ ]:
Code(filename=pbc_pipeline_path, language="YAML")

### Execution

In [ ]:
!qret compile --verbose --pipeline {pbc_pipeline_path}
!qret asm -i {output_dir / "tutorial_5_pbc.json"} -o {output_dir / "tutorial_5_pbc.asm"} --print-metadata 1

Code(filename=output_dir / "tutorial_5_pbc.asm", language="ASM")

## What's Next
In the next chapter, we will analyze the `tutorial_5_*.json` files generated here using the `profile` subcommand. This will allow us to compare key execution resources, such as runtime, code distance, and the required number of physical qubits.
